# ✈️ MODULE 1 : Les Fondations Absolues & Le Modèle Mental de Python
## Projet SkyOps : Système de Télémétrie et Contrôle Aérien

---

Bienvenue dans l'équipe d'ingénierie du noyau de **SkyOps** ! 
Ton rôle est de concevoir le moteur de traitement des données de vol en temps réel.

Ici, nous n'apprenons pas seulement la syntaxe : nous disséquons **comment CPython gère la mémoire**, pourquoi les bugs silencieux surviennent et comment écrire un code natif d'une fiabilité absolue.

> **Règle d'or :** Exécute chaque cellule de cours, lis attentivement les commentaires internes, puis complète les cellules d'exercices jusqu'à ce que les cellules de test affichent le message de validation.

--- 
## 1. Modèle Mémoire : Tout est Objet et Référence (`id`, `is`, `==`)

### 💡 L'Intuition
Dans d'autres langages (C, C++), une variable est une case mémoire physique (une boîte).  
En Python, **une variable n'est jamais une boîte : c'est une étiquette collée avec une ficelle sur un objet existant dans le tas (Heap).**

### ⚙️ Sous le capot (CPython Internals)
- Tout ce qui existe en Python est un `PyObject` en langage C.
- Un `PyObject` minimal contient toujours deux champs critiques :
  1. `ob_refcnt` : le compteur de références (combien de variables pointent vers cet objet).
  2. `ob_type` : un pointeur vers la définition de son type (ex: type `int`, type `str`).
- `id(objet)` retourne l'adresse mémoire directe de ce `PyObject`.
- `a is b` compare les adresses mémoire (`id(a) == id(b)`). C'est un simple test d'identité physique.
- `a == b` vérifie l'égalité de valeur en appelant la méthode interne `a.__eq__(b)`.

In [16]:
# --- DÉMONSTRATION EN DIRECT DE LA MÉMOIRE ---

# 1. Création d'un premier objet de télémétrie (une liste de coordonnées GPS)
radar_paris = [48.8566, 2.3522]

# 2. On affecte une nouvelle étiquette : AUCUNE COPIE N'EST RÉALISÉE
ecran_tour_controle = radar_paris

print(f"Adresse radar_paris          : {hex(id(radar_paris))}")
print(f"Adresse ecran_tour_controle  : {hex(id(ecran_tour_controle))}")
print(f"Pointent-ils vers le même objet ? {radar_paris is ecran_tour_controle}") # True

# 3. Mutation de l'objet partagé
ecran_tour_controle.append(10000)  # On ajoute l'altitude (10 000 m)
print(f"Données vues par radar_paris : {radar_paris}")  # L'étiquette d'origine voit la modification !

Adresse radar_paris          : 0x103af0a80
Adresse ecran_tour_controle  : 0x103af0a80
Pointent-ils vers le même objet ? True
Données vues par radar_paris : [48.8566, 2.3522, 10000]


### ⚠️ Le Piège Classique : Copie superficielle vs Référence
Faire `liste_b = liste_a` ne duplique pas les données. Si tu modifies `liste_b`, `liste_a` est modifiée à ton insu.

### 🎯 EXERCICE 1 : Gestion des Radars de Secours
**Mission :** Un nouveau radar doit cloner les données initiales du vol sans impacter le système principal.

In [17]:
# ====================================================================
# EXERCICE 1 : À COMPLÉTER
# ====================================================================

# Données initiales du vol AF1234 : [cap, vitesse_noeuds, altitude_pieds]
vol_principal = [270, 450, 32000]

# 1. Crée 'moniteur_direct' qui pointe VERS LE MÊME OBJET que vol_principal
moniteur_direct = vol_principal  # <-- Remplace None

# 2. Crée 'simulateur_secours' qui contient les mêmes valeurs mais est UN OBJET DISTINCT
# Astuce : utilise le slicing [:] ou la méthode .copy()
# import copy; copy.deepcopy(element_original)
simulateur_secours = vol_principal.copy()  # <-- Remplace None

# 3. Modifie l'altitude dans simulateur_secours en la passant à 35000
# (vol_principal ne doit pas être affecté)
# Ton code ici :
simulateur_secours[2] = 35000
moniteur_direct[2] = 30000

In [18]:
print(f"Adresse vol_principal          : {hex(id(vol_principal))}")
print(f"Adresse moniteur_direct : {hex(id(moniteur_direct))}")
print(f"Adresse simulateur_secours: {hex(id(simulateur_secours))}")

print(f"Pointent-ils vers le même objet ? {vol_principal is moniteur_direct}") # True
print(f"Pointent-ils vers le même objet ? {vol_principal is simulateur_secours}") # False

# 3. Mutation de l'objet partagé
print(f"Données vues par vol_principal : {vol_principal}") 
print(f"Données vues par  moniteur_direct : { moniteur_direct}") 
print(f"Données vues par simulateur_secours : {simulateur_secours}") 

Adresse vol_principal          : 0x1037aa1c0
Adresse moniteur_direct : 0x1037aa1c0
Adresse simulateur_secours: 0x103b07f80
Pointent-ils vers le même objet ? True
Pointent-ils vers le même objet ? False
Données vues par vol_principal : [270, 450, 30000]
Données vues par  moniteur_direct : [270, 450, 30000]
Données vues par simulateur_secours : [270, 450, 35000]


In [19]:
# ====================================================================
# TESTS DE VALIDATION AUTOMATIQUE (NE PAS MODIFIER)
# ====================================================================
assert moniteur_direct is vol_principal, "❌ moniteur_direct doit être le MÊME objet que vol_principal (is)."
assert simulateur_secours is not vol_principal, "❌ simulateur_secours doit être un objet INDÉPENDANT en mémoire."
assert simulateur_secours[2] == 35000, "❌ L'altitude du simulateur_secours doit être 35000."
assert vol_principal[2] == 32000, "❌ L'altitude du vol_principal ne devait pas bouger (reste à 32000) !"

print("✨ [TEST 1 VALIDÉ] : Modèle d'assignation et références mémoire assimilé avec brio !")

AssertionError: ❌ L'altitude du vol_principal ne devait pas bouger (reste à 32000) !

> 📚 **Référence documentaire & Veille technique :**  
> Pour la gestion des copies d'objets et des adresses mémoire, l'implémentation s'appuie sur la spécification officielle du module standard :  
> [Python Documentation — Module `copy` (Opérations de copie superficielle et profonde)](https://docs.python.org/fr/3.8/library/copy.html)

---
## 2. Mutabilité vs Immutabilité : La Garantie d'Intégrité

### 💡 L'Intuition
- Un objet **Mutable** (ex: `list`, `dict`, `set`) est un classeur dont on peut changer les feuilles sans changer le classeur.
- Un objet **Immutable** (ex: `int`, `float`, `str`, `tuple`, `frozenset`, `bytes`) est un bloc de marbre scellé : impossible de modifier son contenu une fois taillé. Toute prétendue "modification" crée en réalité **un tout nouvel objet** à une autre adresse mémoire.

### ⚙️ Mécanique Interne (Small Integer Caching & Interning)
CPython optimise la mémoire pour les entiers fréquents entre `-5` et `256` : ils sont pré-alloués au démarrage de l'interpréteur.
Toute variable prenant la valeur `42` pointera exactement vers le même `PyObject` unique.

In [ ]:
# Preuve de l'immutabilité des entiers et des chaînes
code_vol = "AF1234"
id_origine = id(code_vol)

# Concaténation : on tente de le 'modifier'
code_vol += "-RETARDE"
id_apres = id(code_vol)

print(f"Ancienne adresse de chaîne : {hex(id_origine)}")
print(f"Nouvelle adresse de chaîne : {hex(id_apres)}")
print(f"Est-ce le même objet ? {id_origine == id_apres}")  # False : un nouvel objet a été instancié !

### ⚠️ Le Piège Dangereux : Le Tuple contenant un objet mutable
Un tuple est immutable au sens où ses *pointeurs internes* ne peuvent plus changer. Mais si l'un de ces pointeurs pointe vers une liste, la liste elle-même reste modifiable !

### 🎯 EXERCICE 2 : Verrouillage de la Boîte Noire
**Mission :** Crée un enregistrement d'identification d'aéronef sécurisé sous forme de `tuple` contenant : l'identifiant (chaîne), le modèle (chaîne) et la liste des réacteurs (liste de booléens `[True, True]`).
Démontre ensuite que tu peux altérer l'état d'un réacteur sans jamais briser l'immutabilité du tuple lui-même.

In [ ]:
# ====================================================================
# EXERCICE 2 : À COMPLÉTER
# ====================================================================

# 1. Définis un tuple nommé 'boite_noire' contenant exactement :
#    - l'indicatif : "AIRBUS-A320"
#    - le statut des réacteurs : une LISTE [True, True] (les deux moteurs tournent)
boite_noire = None  # <-- Remplace None

# 2. Un oiseau percute le moteur 2 : passe le deuxième élément de la liste de réacteurs à False
# Ton code ici :

In [ ]:
# ====================================================================
# TESTS DE VALIDATION AUTOMATIQUE (NE PAS MODIFIER)
# ====================================================================
assert isinstance(boite_noire, tuple), "❌ boite_noire doit impérativement être un tuple."
assert len(boite_noire) == 2, "❌ Le tuple doit contenir 2 éléments : (indicatif, reacteurs)."
assert boite_noire[0] == "AIRBUS-A320", "❌ L'indicatif doit rester 'AIRBUS-A320'."
assert isinstance(boite_noire[1], list), "❌ Le second élément doit être une liste mutable."
assert boite_noire[1] == [True, False], "❌ Le moteur n°2 devrait être éteint [True, False]."

print("✨ [TEST 2 VALIDÉ] : Compréhension chirurgicale des tuples et de la mutabilité !")

---
## 3. L'Anti-Pattern Suprême de Python : L'Argument par Défaut Mutable

### 💡 L'Intuition
Imagine un formulaire d'enregistrement de passagers où la liste de bagages par défaut est pré-remplie avec un panier partagé dans le bureau d'accueil. Si le passager A ne fournit rien, ses bagages vont dans le panier. Si le passager B n'en fournit pas non plus, il se retrouve avec les bagages du passager A !

### ⚙️ Mécanique Interne (Bytecode & Définition de fonction)
En CPython, l'instruction `def` est une **instruction exécutable**. 
Les arguments par défaut sont évalués **UNE SEULE FOIS**, à la compilation/lecture de la fonction par l'interpréteur, et stockés dans l'attribut `__defaults__` de l'objet fonction.
Si tu mets une liste `[]` en valeur par défaut, tous les appels de fonction sans argument partageront **physiquement la même liste en mémoire** !

In [ ]:
# ❌ LE PIÈGE : L'horreur que commettent 90% des développeurs
def enregistrer_alerte_bugge(message, registre=[]):  # Le '[]' est instancié UNE SEULE FOIS
    registre.append(message)
    return registre

print("Vol 101 :", enregistrer_alerte_bugge("Turbulence"))
print("Vol 202 :", enregistrer_alerte_bugge("Carburant bas")) 
# CATASTROPHE : Le Vol 202 hérite de la turbulence du Vol 101 !

In [ ]:
# ✅ LE CODE PYTHONIQUE IDIOMATIQUE : Utiliser 'None' comme sentinelle
def enregistrer_alerte_robuste(message, registre=None):
    if registre is None:
        registre = []  # Une NOUVELLE liste est instanciée à chaque appel spécifique
    registre.append(message)
    return registre

print("Vol 101 :", enregistrer_alerte_robuste("Turbulence"))
print("Vol 202 :", enregistrer_alerte_robuste("Carburant bas"))
# Impeccable : étanchéité parfaite de la mémoire.

### 🎯 EXERCICE 3 : Le Journal de Bord des Vols (Flight Log)
**Mission :** Écris la fonction pythonique `ajouter_escale(nom_escale, plan_de_vol=None)`.
Si aucun plan de vol n'est passé en argument, la fonction doit instancier un nouveau plan de vol (liste) contenant cette seule escale.
Si un plan de vol existant est fourni, l'escale y est ajoutée. La fonction doit toujours renvoyer la liste.

In [ ]:
# ====================================================================
# EXERCICE 3 : À COMPLÉTER
# ====================================================================

def ajouter_escale(nom_escale, plan_de_vol=None):
    """
    Ajoute une escale à un plan de vol sans risque de fuite mémoire.
    """
    # --- À TOI DE CODER ICI ---
    pass

In [ ]:
# ====================================================================
# TESTS DE VALIDATION AUTOMATIQUE (NE PAS MODIFIER)
# ====================================================================
vol_a = ajouter_escale("Paris CDG")
vol_b = ajouter_escale("Londres LHR")

assert vol_a == ["Paris CDG"], f"❌ vol_a devrait contenir ['Paris CDG'], mais contient : {vol_a}"
assert vol_b == ["Londres LHR"], f"❌ vol_b a été pollué par vol_a ! Contient : {vol_b}"
assert vol_a is not vol_b, "❌ vol_a et vol_b partagent la même adresse mémoire ! Utilise 'None' comme sentinelle."

# Test avec plan existant
mon_trajet = ["Nice"]
vol_c = ajouter_escale("Rome", mon_trajet)
assert vol_c is mon_trajet, "❌ Lorsque le plan est fourni, il doit être retourné après modification."
assert vol_c == ["Nice", "Rome"], "❌ Le plan de vol devrait être ['Nice', 'Rome']."

print("✨ [TEST 3 VALIDÉ] : L'anti-pattern le plus vicieux de Python est vaincu !")

---
## 4. Dictionnaires & Hash Maps : Accès $O(1)$ et Déballage Idiomatique

### 💡 L'Intuition
Chercher un vol par son numéro dans une liste de 10 000 éléments oblige à parcourir toute la liste ($O(N)$).  
Un dictionnaire utilise une table de hachage (**Hash Table**) : il applique une fonction mathématique (`hash()`) sur la clé pour sauter directement à la bonne adresse mémoire en une seule opération ($O(1)$).

### ⚙️ Règle Absolue de CPython
Une clé de dictionnaire **DOIT ÊTRE HASHABLE** (c'est-à-dire immutable avec une méthode `__hash__`).
- Valides comme clés : `int`, `str`, `tuple` (ne contenant que des éléments immutables).
- Invalides : `list`, `dict`, `set` (déclencheront un `TypeError: unhashable type`).

In [ ]:
# Démonstration : la fonction hash()
print("Hash de 'AF1234' :", hash("AF1234"))
print("Hash du tuple (48.85, 2.35) :", hash((48.85, 2.35)))

try:
    # Tentative d'utiliser une liste comme clé
    liste_cle = [1, 2]
    mauvais_dict = {liste_cle: "interdit"}
except TypeError as e:
    print("Erreur interceptée par CPython :", e)

### 🎯 EXERCICE 4 : L'Aiguilleur du Ciel & Déballage (Unpacking)
**Mission :** 
1. Crée un dictionnaire `vols_en_cours` dont la clé est le tuple de coordonnées de départ `(latitude, longitude)` et la valeur est l'indicatif du vol (str).
   - Associe `(43.66, 7.21)` (Nice) à `"AF1234"`.
   - Associe `(48.85, 2.35)` (Paris) à `"BA567"`.
2. En utilisant l'idiome `.get()`, récupère de façon sécurisée le vol aux coordonnées `(51.50, -0.12)` (Londres) avec la valeur de repli par défaut `"INCONNU"` dans une variable `vol_londres`.
3. Fusionne ces vols avec un second dictionnaire `nouveaux_vols = {(52.31, 4.76): "KLM88"}` dans un tout nouveau dictionnaire nommé `flotte_complete` en utilisant l'opérateur de fusion dictionnaire moderne `|` (Python 3.9+).

In [ ]:
# ====================================================================
# EXERCICE 4 : À COMPLÉTER
# ====================================================================

# 1. Initialise vols_en_cours
vols_en_cours = None  # <-- Remplace None

# 2. Récupère vol_londres avec .get()
vol_londres = None  # <-- Remplace None

# 3. Fusion avec nouveaux_vols
nouveaux_vols = {(52.31, 4.76): "KLM88"}
flotte_complete = None  # <-- Remplace None

In [ ]:
# ====================================================================
# TESTS DE VALIDATION AUTOMATIQUE (NE PAS MODIFIER)
# ====================================================================
assert vols_en_cours[(43.66, 7.21)] == "AF1234", "❌ Coordonnées de Nice erronées."
assert vols_en_cours[(48.85, 2.35)] == "BA567", "❌ Coordonnées de Paris erronées."
assert vol_londres == "INCONNU", f"❌ vol_londres devrait être 'INCONNU', obtenu : {vol_londres}"
assert len(flotte_complete) == 3, "❌ flotte_complete doit contenir 3 éléments après fusion."
assert flotte_complete[(52.31, 4.76)] == "KLM88", "❌ Le vol KLM88 est manquant dans la flotte complète."

print("✨ [TEST 4 VALIDÉ] : Dictionnaires et Hashing maîtrisés avec élégance !")

---
## 🏆 BILAN DU NIVEAU 1 & VALIDATION DU MODULE

Si toutes les cellules ci-dessus ont affiché leur message de succès : 
Tu as posé **les vraies fondations de la mémoire Python** :
1. Tu visualises les variables comme des étiquettes et non des boîtes.
2. Tu distingues l'identité mémoire (`is`) de l'égalité de valeur (`==`).
3. Tu maîtrises la mutabilité et sais prémunir ton code du piège de l'argument par défaut mutable.
4. Tu comprends le contrat de hachabilité des dictionnaires.

Transmets ton code d'exercice résolu à ton mentor pour passer au **NIVEAU 2 : Python Idiomatique & POO Fondamentale (Dunder methods, Protocoles & Générateurs)** !